In [ ]:
!pip install pyECLAT
from pyECLAT import ECLAT
import pandas as pd

df = pd.read_csv("/content/products.csv")
df.head()

,TransactionID,CustomerID,Products,Timestamp
0,1,C546,"Dish Sponge, Flatbread with Meat, Chips, Orang...",2025-02-18
1,2,C385,"Onion, Juice, Flatbread with Meat, Chicken",2025-04-26
2,3,C292,"Egg, Flatbread with Meat, Banana, Pizza",2025-04-25
3,4,C863,"Ice Cream, Soda, Orange, Potato, Cereal, Choco...",2025-01-14
4,5,C171,"Ice Cream, Soap, Shampoo, Chicken, Banana, Bea...",2025-04-20


In [26]:
df.isna().sum()

,0
TransactionID,0
CustomerID,0
Products,0
Timestamp,0


In [ ]:
transactions = df['Products'].str.split(',', expand=True)

transactions = transactions.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(transactions.head())

eclat = ECLAT(data=transactions)

indices, support = eclat.fit(
    min_support=0.05,
    min_combination=1,
    max_combination=3
)

print("\nFrequent Itemsets:")
print(support)

            0                    1                    2        3       4   \
0  Dish Sponge  Flatbread with Meat                Chips   Orange  Butter   
1        Onion                Juice  Flatbread with Meat  Chicken    None   
2          Egg  Flatbread with Meat               Banana    Pizza    None   
3    Ice Cream                 Soda               Orange   Potato  Cereal   
4    Ice Cream                 Soap              Shampoo  Chicken  Banana   

          5         6     7          8     9     10  
0    Sausage  Cucumber  Rice  Ice Cream  None  None  
1       None      None  None       None  None  None  
2       None      None  None       None  None  None  
3  Chocolate   Cracker  None       None  None  None  
4      Beans    Cheese  None       None  None  None  
Combination 1 by 1


40it [00:01, 20.53it/s]


Combination 2 by 2


780it [00:07, 107.01it/s]


Combination 3 by 3


9880it [01:36, 102.38it/s]


Frequent Itemsets:
{'Lentil': 0.1617, 'Yogurt': 0.16376666666666667, 'Shampoo': 0.16176666666666667, 'Soap': 0.16506666666666667, 'Bread': 0.16233333333333333, 'Beans': 0.16496666666666668, 'Water': 0.15996666666666667, 'Flatbread with Meat': 0.16263333333333332, 'Cucumber': 0.15993333333333334, 'Chickpeas': 0.1637, 'Juice': 0.1653, 'Honey': 0.1612, 'Potato': 0.16206666666666666, 'Rice': 0.16266666666666665, 'Dish Sponge': 0.16256666666666666, 'Orange': 0.16483333333333333, 'Cookie': 0.16393333333333332, 'Tomato': 0.16256666666666666, 'Chips': 0.1617, 'Milk': 0.16266666666666665, 'Apple': 0.16046666666666667, 'Banana': 0.16303333333333334, 'Onion': 0.1624, 'Chocolate': 0.1608, 'Detergent': 0.15736666666666665, 'Cracker': 0.16236666666666666, 'Soda': 0.1658, 'Egg': 0.1621, 'Ice Cream': 0.16793333333333332, 'Chicken': 0.16603333333333334, 'Cheese': 0.16526666666666667, 'Minced Meat': 0.1631, 'Fish': 0.1636, 'Cereal': 0.23273333333333332, 'Strawberry': 0.16346666666666668, 'Pizza': 0.159

In [ ]:
support_df = pd.DataFrame(
    support.items(),
    columns=['Itemset', 'Support']
)

support_df = support_df.sort_values(
    by='Support',
    ascending=False
)

print(support_df.head(20))

        Itemset   Support
33       Cereal  0.232733
28    Ice Cream  0.167933
29      Chicken  0.166033
26         Soda  0.165800
10        Juice  0.165300
30       Cheese  0.165267
3          Soap  0.165067
5         Beans  0.164967
15       Orange  0.164833
39      Sausage  0.164400
16       Cookie  0.163933
1        Yogurt  0.163767
9     Chickpeas  0.163700
32         Fish  0.163600
34   Strawberry  0.163467
36         Cola  0.163367
31  Minced Meat  0.163100
21       Banana  0.163033
13         Rice  0.162667
19         Milk  0.162667


In [25]:
!pip install mlxtend -q

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_csv('/content/products.csv')
df.head()

,TransactionID,CustomerID,Products,Timestamp
0,1,C546,"Dish Sponge, Flatbread with Meat, Chips, Orang...",2025-02-18
1,2,C385,"Onion, Juice, Flatbread with Meat, Chicken",2025-04-26
2,3,C292,"Egg, Flatbread with Meat, Banana, Pizza",2025-04-25
3,4,C863,"Ice Cream, Soda, Orange, Potato, Cereal, Choco...",2025-01-14
4,5,C171,"Ice Cream, Soap, Shampoo, Chicken, Banana, Bea...",2025-04-20


In [27]:
df.isna().sum()

,0
TransactionID,0
CustomerID,0
Products,0
Timestamp,0


In [24]:
print("Columns:", df.columns.tolist())

transactions = df['Products'].dropna().apply(
    lambda x: [item.strip() for item in str(x).split(',')]
).tolist()

print("Number of Transactions:", len(transactions))


Columns: ['TransactionID', 'CustomerID', 'Products', 'Timestamp']
Number of Transactions: 30000


In [28]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(te_array, columns=te.columns_)

print("\nEncoded Dataset Shape:")
print(basket.shape)



Encoded Dataset Shape:
(30000, 40)


In [30]:
frequent_itemsets = fpgrowth(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("\nFrequent Itemsets:")
print(frequent_itemsets.head(20))



Frequent Itemsets:
     support               itemsets
0   0.167933            (Ice Cream)
1   0.164833               (Orange)
2   0.164400              (Sausage)
3   0.162667                 (Rice)
4   0.162633  (Flatbread with Meat)
5   0.162567          (Dish Sponge)
6   0.162033               (Butter)
7   0.161700                (Chips)
8   0.159933             (Cucumber)
9   0.166033              (Chicken)
10  0.165300                (Juice)
11  0.162400                (Onion)
12  0.163033               (Banana)
13  0.162100                  (Egg)
14  0.159433                (Pizza)
15  0.232733               (Cereal)
16  0.165800                 (Soda)
17  0.162367              (Cracker)
18  0.162067               (Potato)
19  0.160800            (Chocolate)


In [31]:
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.2
)

# Sort by Lift
rules = rules.sort_values(
    by='lift',
    ascending=False
)


In [32]:
print("\nTop 20 Association Rules:")
print(
    rules[
        ['antecedents',
         'consequents',
         'support',
         'confidence',
         'lift']
    ].head(20)
)


Top 20 Association Rules:
    antecedents consequents   support  confidence      lift
33     (Cereal)      (Milk)  0.095067    0.408479  2.511141
32       (Milk)    (Cereal)  0.095067    0.584426  2.511141
18  (Chocolate)    (Cereal)  0.038933    0.242123  1.040344
8    (Cucumber)    (Cereal)  0.038600    0.241351  1.037026
19     (Cheese)    (Cereal)  0.039733    0.240420  1.033026
6      (Butter)    (Cereal)  0.038733    0.239045  1.027122
15       (Soda)    (Cereal)  0.039400    0.237636  1.021064
11      (Onion)    (Cereal)  0.038433    0.236658  1.016865
7       (Chips)    (Cereal)  0.038233    0.236446  1.015953
27     (Tomato)    (Cereal)  0.038300    0.235596  1.012299
23       (Fish)    (Cereal)  0.038533    0.235534  1.012033
10      (Juice)    (Cereal)  0.038900    0.235330  1.011156
24      (Bread)    (Cereal)  0.038167    0.235113  1.010225
17     (Potato)    (Cereal)  0.038000    0.234471  1.007468
21      (Beans)    (Cereal)  0.038667    0.234391  1.007122
0   (Ice Crea

In [33]:
rules.to_csv(
    '/content/drive/MyDrive/DATASETS/fpgrowth_rules.csv',
    index=False
)

print("\nRules saved successfully.")


Rules saved successfully.
